In [3]:
from dotenv import load_dotenv
load_dotenv()


True

In [4]:
from openai import OpenAI
openai_client = OpenAI()

In [5]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
llm("Hey, what's up?")

'Hey! Not much—just here and ready to help. What can I do for you?'

In [7]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely — in most cases, yes, you can join now.

If you’ve just discovered the course, the next step is usually to check:
- whether enrollment is still open,
- whether there’s a late-start option or self-paced access,
- and any deadlines for assignments or certificates.

If you want, I can help you figure out the best way to ask the course organizer or draft a short message like:

> Hi, I just found the course and I’m very interested in joining. Is it still possible to enroll at this point?

If you tell me the course name or platform, I can give more specific guidance.


In [8]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [9]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [10]:
answer = llm(prompt)
print(answer)

Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.


In [11]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [12]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [13]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 482},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 161},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 398},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 73},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 249}]

In [14]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1456

In [15]:
documents[90]

{'id': '0a4f9065b1',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 2. Machine Learning for Regression',
 'question': 'Loading the dataset directly through Kaggle Notebooks',
 'answer': "To load a dataset in Kaggle Notebooks, you can use the following command. Remember that the `!` before `wget` is essential.\n\n```bash\n!wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/housing.csv\n```\n\nOnce the dataset is loaded onto the Kaggle Notebook server, it can be read using the following pandas command:\n\n```python\ndf = pd.read_csv('housing.csv')\n```"}

In [16]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [17]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [18]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [19]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Does the course certificate show the number of course hours?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?']

In [20]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search("what is criteria to get certificate")

In [22]:
search_results

[{'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'febc213623',
  'course': 'llm-zoomcamp',
  'section': 'Capstone Project',
  'question': 'I passed the project. When will I get my certificate?',
  'answer': 'You\'ll get it after the last project attempt and its peer review are over.\nCertificates are generated manually, in one batch for all graduates of the\ncohort.\n\nSo if you passed an earlier attempt, wait until the remaining attempts 

In [23]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [24]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [25]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [26]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

To get the certificate, you need to finish a capstone project and complete the
required peer reviews. Homework is not required. You can work through the
material and prepare your project in self-paced mode, but project submission and
peer review must happen while a live cohort is accepting them.

Capstone Project
Q: I passed the project. When will I get my certificate?
A: You'll get it after the last project attempt and its peer review are over.
Certificates are generated manually, in one batch for all graduates of the
cohort.

So if you passed an earlier attempt, wait until the remaining attempts are
finished. You don't need to request the certificate. Before then, make sure the
certificate name in your profile on the
[course management platform](https://courses.datatalks.club/llm-zoom

In [27]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [28]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

To get the certificate, you need to finish a capstone project and complete the
required peer reviews. Homework is not required. You can work through the
material and prepare your project in self-paced mode, but project submission and
peer review must happen while a live cohort is accepting them.

Capstone Project
Q: I passed the project. When will I get my certificate?
A: You'll get it after the last project attempt and its peer review are over.
Certificates are generated manually, in one batch for all graduates of the
cohort.

So if you passed an earlier attempt, wait until the remaining attempts are
finished. You don't need to request the certificate. Before then, make sure the
certificate name in your profile on the
[c

In [29]:
llm(prompt)

'Yes — you can join now and work through the materials in self-paced mode.\n\nA couple of important notes:\n\n- You can **study anytime**.\n- To get a **certificate**, you must complete the **capstone project** and the required **peer reviews** **while a live cohort is accepting submissions**.\n- **Homework is not required** for the certificate.\n\nSo: feel free to start now, but make sure you’re active in the live cohort when it’s time to submit the project and do peer reviews.'

In [30]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [31]:
response.output[0]

ResponseOutputMessage(id='msg_00127e59827eeaca006aeb45df299c87d284702452302e48e4', content=[ResponseOutputText(annotations=[], text='Yes — you can join, but it depends on whether the cohort is still open.\n\nA few important points:\n\n- You can work through the course material in self-paced mode.\n- To get a certificate, you must join and complete the **live cohort** requirements.\n- The certificate requires:\n  - passing the **capstone project**,\n  - completing the required **peer reviews**.\n\nSo if the current cohort is still accepting participants/submissions, you can join now and still be eligible. If the cohort has already closed, you can still study the material, but you won’t be able to get a certificate from that cohort.\n\nCheck the course dashboard or platform to confirm whether the live cohort is still active.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')

In [32]:
response.output[0].content[0].text


'Yes — you can join, but it depends on whether the cohort is still open.\n\nA few important points:\n\n- You can work through the course material in self-paced mode.\n- To get a certificate, you must join and complete the **live cohort** requirements.\n- The certificate requires:\n  - passing the **capstone project**,\n  - completing the required **peer reviews**.\n\nSo if the current cohort is still accepting participants/submissions, you can join now and still be eligible. If the cohort has already closed, you can still study the material, but you won’t be able to get a certificate from that cohort.\n\nCheck the course dashboard or platform to confirm whether the live cohort is still active.'

In [33]:
response.output_text


'Yes — you can join, but it depends on whether the cohort is still open.\n\nA few important points:\n\n- You can work through the course material in self-paced mode.\n- To get a certificate, you must join and complete the **live cohort** requirements.\n- The certificate requires:\n  - passing the **capstone project**,\n  - completing the required **peer reviews**.\n\nSo if the current cohort is still accepting participants/submissions, you can join now and still be eligible. If the cohort has already closed, you can still study the material, but you won’t be able to get a certificate from that cohort.\n\nCheck the course dashboard or platform to confirm whether the live cohort is still active.'

In [34]:
response.usage

ResponseUsage(input_tokens=688, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=147, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=835)

In [35]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.0011775

In [36]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [38]:
def llm_new(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [40]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm_new(INSTRUCTIONS, prompt, model=model)
    return answer

In [41]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes — you can still join now. If you want a certificate, make sure to submit your project while the course is still accepting submissions.


In [42]:
rag("How do I get a certificate?")


'You can get a certificate only if you finish the course with a live cohort.\n\nTo qualify, you need to:\n- complete the capstone project\n- finish the required peer reviews\n\nHomework is not required.\n\nIf you pass the project, the certificate is issued after the last project attempt and its peer review are over. Certificates are generated manually in one batch, and you’ll get an email when it’s ready.\n\nBefore that, make sure the name in your course profile is exactly how you want it printed on the certificate.'